# Stage 0 — Environment & Repo Setup

Repo scaffold check + smoke test for all 4 base models in 4-bit on T4:
Gemma-2-2B-IT, Llama-3.2-3B-Instruct, Qwen2.5-3B-Instruct, Phi-4-mini-instruct.

Follows the rules in `claude.md` — Drive mount, repo clone, anti-disconnect, one model on GPU at a time, checkpointed saves.

**Before running:** accept the model licenses on Hugging Face for `google/gemma-2-2b-it` and `meta-llama/Llama-3.2-3B-Instruct` (gated repos), and add your HF token as a Colab secret named `HF_TOKEN`.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes datasets scikit-learn matplotlib

## 1. Mount Drive (claude.md rule 1)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/RepEng_Survival/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

## 2. Clone/pull repo (claude.md rule 1)

In [ ]:
REPO_URL = 'https://github.com/Pranamya16/RepEng_Survival.git'

if not os.path.exists('/content/RepEng_Survival'):
    !git clone {REPO_URL} /content/RepEng_Survival
else:
    !cd /content/RepEng_Survival && git pull

%cd /content/RepEng_Survival

import sys
sys.path.append('/content/RepEng_Survival/src')

## 3. Anti-disconnect (claude.md rule 2)
Keeps the session alive against Colab's idle timeout. Does not bypass the 12-hour hard session limit — that's what checkpointing (below) is for.

In [ ]:
from IPython.display import Javascript

display(Javascript('''
function KeepAlive(){
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(KeepAlive, 60000)
'''))

## 4. Hugging Face login (Gemma-2 and Llama-3.2 are gated)

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(userdata.get('HF_TOKEN'))

## 5. Checkpoint helpers (claude.md rule 4)

In [ ]:
import json

def save_result(name, data):
    path = os.path.join(RESULTS_DIR, name)
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)

def load_result_if_exists(name):
    path = os.path.join(RESULTS_DIR, name)
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

## 6. Smoke test — one model at a time (claude.md rule 3)
Loads each model in 4-bit, generates one short response, saves the result to Drive, then fully unloads before moving to the next model. Resumable: already-completed models are skipped.

In [ ]:
import torch
import gc
import traceback
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODELS = {
    'gemma-2-2b-it': 'google/gemma-2-2b-it',
    'llama-3.2-3b-instruct': 'meta-llama/Llama-3.2-3B-Instruct',
    'qwen2.5-3b-instruct': 'Qwen/Qwen2.5-3B-Instruct',
    'phi-4-mini-instruct': 'microsoft/Phi-4-mini-instruct',
}

SMOKE_PROMPT = 'In one sentence, explain what a neural network is.'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

for short_name, hf_id in MODELS.items():
    result_name = f'stage0_smoketest_{short_name}.json'
    existing = load_result_if_exists(result_name)
    if existing and existing.get('status') == 'ok':
        print(f'Skipping {short_name}, already done.')
        continue

    print(f'Loading {short_name} ({hf_id}) ...')
    model = None
    tokenizer = None
    try:
        tokenizer = AutoTokenizer.from_pretrained(hf_id)
        model = AutoModelForCausalLM.from_pretrained(
            hf_id, quantization_config=bnb_config, device_map='auto'
        )

        messages = [{'role': 'user', 'content': SMOKE_PROMPT}]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors='pt'
        ).to(model.device)
        output = model.generate(inputs, max_new_tokens=60, do_sample=False)
        generated = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)

        result = {
            'model': short_name,
            'hf_id': hf_id,
            'status': 'ok',
            'sample_generation': generated,
        }
        print(generated)

    except Exception as e:
        tb = traceback.format_exc()
        result = {'model': short_name, 'hf_id': hf_id, 'status': 'error', 'error': str(e) or repr(e), 'traceback': tb}
        print(f'FAILED: {e!r}')
        print(tb)

    save_result(result_name, result)

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

## 7. Results stay on Drive (claude.md rule 5)
Results were saved to `RESULTS_DIR` on Drive as each model finished. Nothing here gets pushed to GitHub — only code changes to `/src` or `/notebooks` get pushed, separately.

## Deliverable check
- [ ] Repo scaffold present (this notebook + `/src` cloned successfully)
- [ ] All 4 models loaded in 4-bit and generated a coherent response
- [ ] Results saved to Drive (`RESULTS_DIR`) — not pushed to GitHub